# MIMIC-IV in-hospital mortality: data preparation

Builds the cohort, labels, event sequences, tokenizer and LLM text used by every model notebook
(`02_gpt2_mamba`, `03_gpt2_ar`, `04_qwen`), so all models share the same stays and the same
train / validation / test split.

**Task definition** (same as the CAREBench `transformers/mortality_*` and `llm/tasks/mimic_mortality` code)
* One ICU stay per patient. For patients who died in hospital, it's the last ICU stay of the admission in which they died. For patients who never died in hospital, it's their last ICU stay.
* `end_time` = `deathtime` if the patient died in hospital, otherwise ICU `outtime`.
* **Prediction time** = `end_time - 12 h`. Model inputs are all events from ICU admission up to the prediction time.
* **Label** = 1 if the patient died in hospital (i.e. died 12 h after the prediction time).

**Events.** Numeric values from `chartevents`, `labevents`, `inputevents`, `outputevents`, `procedureevents`
and `ingredientevents` become tokens of the form `itemid|low-high`. The bucket edges are per-item deciles
computed on the training split, as in `process_data/mimic-mortality.ipynb`.

**Required MIMIC-IV v3.1 files** (`.csv.gz` as downloaded from PhysioNet, or decompressed `.csv`) under `MIMIC_DIR`:
`hosp/{admissions,patients,labevents,d_labitems}` and `icu/{icustays,chartevents,inputevents,outputevents,procedureevents,ingredientevents,d_items}`.

Runtime: `chartevents` has ~430M rows, so the full extraction takes 1–2 h and needs ~16 GB RAM. Every step writes
its output to disk and is skipped when re-run. Set `SUBSAMPLE_PATIENTS` for a quick dry run.

In [ ]:
import sys
import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd()))
from mortality_common import MIMIC_DIR, DATA_DIR, PREDICTION_GAP_HOURS, SPECIAL_TOKENS

In [ ]:
# ---------------- configuration ----------------
# MIMIC_DIR / DATA_DIR default to <repo>/data/mimiciv/3.1 and <repo>/data/mimic_mortality
# (override with the MIMIC_DIR / MORTALITY_DATA_DIR environment variables or here).
MIMIC_DIR = Path(MIMIC_DIR)
OUT_DIR = Path(DATA_DIR)

SEED = 1234
SUBSAMPLE_PATIENTS = None          # e.g. 3000 for a quick end-to-end dry run; None = full cohort
MAX_DEATH_AFTER_ICU_HOURS = 24     # positives must die within this many hours of ICU discharge (None = keep all)
MIN_OBS_HOURS = 0.0                # prediction time must be at least this long after ICU admission
MIN_EVENTS = 5                     # drop stays with fewer events before the prediction time
MAX_EVENTS = 2048                  # keep the most recent N events per stay
N_BUCKETS = 10                     # decile buckets per numeric item
BUCKET_SAMPLES_PER_PARTITION = 2000  # values sampled per item per partition to estimate deciles
N_PARTITIONS = 32                  # stays are hashed into partitions so each fits in memory
CSV_CHUNK_SIZE = 5_000_000

# Text serialisation for the LLM (matches llm/tasks/mimic_mortality/data/gen_dataset.py)
LLM_LOOKBACK_HOURS = 36            # last 1.5 days before the prediction time
LLM_RESAMPLE_MINUTES = 30          # keep the first value of each item per 30-minute bin
LLM_MAX_ROWS = 700                 # most recent measurements per stay
LLM_SPLITS = ['test']              # add 'val' to also serialise the validation stays

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('MIMIC_DIR:', MIMIC_DIR.resolve())
print('OUT_DIR  :', OUT_DIR.resolve())


def mimic_file(rel):
    """Resolve hosp/admissions -> .csv.gz or .csv."""
    for ext in ['.csv.gz', '.csv']:
        path = MIMIC_DIR / f'{rel}{ext}'
        if path.exists():
            return path
    raise FileNotFoundError(f'{rel}.csv(.gz) not found under {MIMIC_DIR}')

## 1. Cohort and labels

In [ ]:
TIME_FMT = '%Y-%m-%d %H:%M:%S'

adm = pd.read_csv(mimic_file('hosp/admissions'), usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'race'])
pat = pd.read_csv(mimic_file('hosp/patients'), usecols=['subject_id', 'gender', 'anchor_age', 'anchor_year'])
icu = pd.read_csv(mimic_file('icu/icustays'), usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime'])
for df, cols in [(adm, ['admittime', 'dischtime', 'deathtime']), (icu, ['intime', 'outtime'])]:
    for c in cols:
        df[c] = pd.to_datetime(df[c], format=TIME_FMT)

icu = icu.merge(adm, on=['subject_id', 'hadm_id'], how='inner').dropna(subset=['intime', 'outtime'])
died_in_hospital = set(adm.loc[adm['deathtime'].notna(), 'subject_id'])

# positives: last ICU stay of the admission in which the patient died
pos = icu[icu['deathtime'].notna()].sort_values('intime').groupby('subject_id').tail(1).copy()
pos['label'] = 1
# negatives: patients who never died in hospital, last ICU stay
neg = icu[~icu['subject_id'].isin(died_in_hospital)].sort_values('intime').groupby('subject_id').tail(1).copy()
neg['label'] = 0

cohort = pd.concat([pos, neg], ignore_index=True)
cohort['end_time'] = cohort['deathtime'].where(cohort['label'] == 1, cohort['outtime'])
cohort['cutoff'] = cohort['end_time'] - pd.Timedelta(hours=PREDICTION_GAP_HOURS)
print('candidate stays:', len(cohort), '| deaths:', int(cohort['label'].sum()))

keep = cohort['end_time'] >= cohort['intime']
if MAX_DEATH_AFTER_ICU_HOURS is not None:
    # deaths long after ICU discharge would only have ward labs in the window -> easy shortcut for the models
    late_death = (cohort['label'] == 1) & (cohort['deathtime'] > cohort['outtime'] + pd.Timedelta(hours=MAX_DEATH_AFTER_ICU_HOURS))
    print('dropping deaths >', MAX_DEATH_AFTER_ICU_HOURS, 'h after ICU discharge:', int(late_death.sum()))
    keep &= ~late_death
keep &= (cohort['cutoff'] - cohort['intime']).dt.total_seconds() / 3600 >= MIN_OBS_HOURS
cohort = cohort[keep].copy()
print(f'stays with >= {PREDICTION_GAP_HOURS + MIN_OBS_HOURS:.0f} h between ICU admission and end of stay:', len(cohort), '| deaths:', int(cohort['label'].sum()))

### Demographics
Demographic tokens (`Gender/*`, `RACE/*`, `Age/*`) are prepended to every sequence. They use the same vocabulary as `data/mimic_tokenizer.json`.

In [ ]:
AGES = [0, 20, 30, 40, 50, 60, 70, 80, 90, 200]
AGE_LABELS = [f'Age/{AGES[i]}-{AGES[i + 1]}' for i in range(len(AGES) - 2)] + ['Age/90-100']


def race_str(x):
    x = str(x).upper()
    if 'HISPANIC' in x or 'LATINO' in x: return 'RACE/HISPANIC'
    if 'ASIAN' in x: return 'RACE/ASIAN'
    if 'BLACK' in x: return 'RACE/BLACK'
    if 'WHITE' in x: return 'RACE/WHITE'
    if 'NATIVE' in x: return 'RACE/AMERICAN'
    if x == 'PORTUGUESE': return 'RACE/PORTUGUESE'
    if x == 'SOUTH AMERICAN': return 'RACE/SOUTH'
    return 'RACE/UNKNOWN'


def age_str(age):
    return AGE_LABELS[min(np.searchsorted(AGES, age, 'right') - 1, len(AGE_LABELS) - 1)]


cohort = cohort.merge(pat, on='subject_id', how='left')
cohort['age'] = cohort['anchor_age'] + (cohort['intime'].dt.year - cohort['anchor_year'])
cohort['gender_str'] = 'Gender/' + cohort['gender'].fillna('unknown')
cohort['race_str'] = cohort['race'].apply(race_str)
cohort['age_str'] = cohort['age'].fillna(0).apply(age_str)

### Train / validation / test split (60 / 20 / 20, stratified by label, one stay per patient)

In [ ]:
if SUBSAMPLE_PATIENTS is not None and SUBSAMPLE_PATIENTS < len(cohort):
    cohort, _ = train_test_split(cohort, train_size=SUBSAMPLE_PATIENTS, stratify=cohort['label'], random_state=SEED)

train_ids, tmp = train_test_split(cohort['stay_id'], test_size=0.4, stratify=cohort['label'], random_state=SEED)
tmp_labels = cohort.set_index('stay_id').loc[tmp, 'label']
val_ids, test_ids = train_test_split(tmp, test_size=0.5, stratify=tmp_labels, random_state=SEED)
cohort['split'] = 'train'
cohort.loc[cohort['stay_id'].isin(val_ids), 'split'] = 'val'
cohort.loc[cohort['stay_id'].isin(test_ids), 'split'] = 'test'
cohort['partition'] = (cohort['stay_id'] % N_PARTITIONS).astype(int)
cohort = cohort.sort_values('stay_id').reset_index(drop=True)

cohort_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'deathtime', 'end_time', 'cutoff',
               'label', 'split', 'partition', 'age', 'gender_str', 'race_str', 'age_str']
cohort[cohort_cols].to_csv(OUT_DIR / 'cohort.csv', index=False)
print(cohort.groupby('split')['label'].agg(['count', 'sum', 'mean']))

## 2. Extract events within each stay's observation window

Every table is streamed in chunks and filtered to the cohort and to `[intime, cutoff]`. ICU tables must also match the
stay_id; lab events are matched on subject and time, since many have no `hadm_id`. Output goes to `events_raw/part_XXX/`.
A `_DONE_<table>` marker lets an interrupted run resume.

In [ ]:
EVENT_TABLES = {
    # name: (file, time column, value column, has stay_id)
    'chartevents':      ('icu/chartevents',      'charttime', 'valuenum', True),
    'labevents':        ('hosp/labevents',       'charttime', 'valuenum', False),
    'inputevents':      ('icu/inputevents',      'starttime', 'amount',   True),
    'outputevents':     ('icu/outputevents',     'charttime', 'value',    True),
    'procedureevents':  ('icu/procedureevents',  'starttime', 'value',    True),
    'ingredientevents': ('icu/ingredientevents', 'starttime', 'amount',   True),
}
RAW_DIR = OUT_DIR / 'events_raw'
# invalidate the cached extraction if the cohort / windows / partitioning changed since it was written
fingerprint = str(pd.util.hash_pandas_object(cohort[['stay_id', 'cutoff', 'partition']], index=False).sum())
if RAW_DIR.exists() and (RAW_DIR / '_COHORT').exists() and (RAW_DIR / '_COHORT').read_text() != fingerprint:
    print('cohort changed -> re-extracting events')
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(exist_ok=True)
(RAW_DIR / '_COHORT').write_text(fingerprint)

windows = cohort.set_index('subject_id')[['stay_id', 'intime', 'cutoff', 'partition']].rename(columns={'stay_id': 'cohort_stay_id'})


def extract_table(name, rel, time_col, value_col, has_stay):
    done_flag = RAW_DIR / f'_DONE_{name}'
    if done_flag.exists():
        print(f'{name}: already extracted')
        return
    for old in RAW_DIR.glob(f'part_*/{name}_*.parquet'):  # clean a partially written run
        old.unlink()
    usecols = ['subject_id', 'itemid', time_col, value_col] + (['stay_id'] if has_stay else [])
    reader = pd.read_csv(mimic_file(rel), usecols=usecols, chunksize=CSV_CHUNK_SIZE,
                         dtype={value_col: 'float64', 'stay_id': 'float64'})
    n_kept = 0
    for i, chunk in enumerate(tqdm(reader, desc=name, unit='chunk')):
        chunk = chunk[chunk['subject_id'].isin(windows.index)].dropna(subset=[value_col, time_col])
        chunk = chunk.join(windows, on='subject_id')
        if has_stay:
            chunk = chunk[chunk['stay_id'] == chunk['cohort_stay_id']]
        t = pd.to_datetime(chunk[time_col], format=TIME_FMT)
        in_window = (t >= chunk['intime']) & (t <= chunk['cutoff'])
        chunk, t = chunk[in_window], t[in_window]
        if chunk.empty:
            continue
        out = pd.DataFrame({
            'stay_id': chunk['cohort_stay_id'].astype('int64').to_numpy(),
            'itemid': chunk['itemid'].astype('int32').to_numpy(),
            'hours': ((t - chunk['intime']).dt.total_seconds() / 3600).astype('float32').to_numpy(),
            'value': chunk[value_col].astype('float32').to_numpy(),
            'partition': chunk['partition'].to_numpy(),
        })
        for p, part in out.groupby('partition'):
            part_dir = RAW_DIR / f'part_{p:03d}'
            part_dir.mkdir(exist_ok=True)
            part.drop(columns='partition').to_parquet(part_dir / f'{name}_{i:05d}.parquet', index=False)
        n_kept += len(out)
    done_flag.touch()
    print(f'{name}: kept {n_kept:,} events')


for name, spec in EVENT_TABLES.items():
    extract_table(name, *spec)
    gc.collect()


def read_partition(p, columns=None):
    part_dir = RAW_DIR / f'part_{p:03d}'
    files = sorted(part_dir.glob('*.parquet')) if part_dir.exists() else []
    if not files:
        return pd.DataFrame({'stay_id': pd.Series(dtype='int64'), 'itemid': pd.Series(dtype='int32'),
                             'hours': pd.Series(dtype='float32'), 'value': pd.Series(dtype='float32')})[columns or slice(None)]
    return pd.concat([pd.read_parquet(f, columns=columns) for f in files], ignore_index=True)

## 3. Per-item decile buckets (training split only)

Same scheme as `valuenum()` / `bucket_eventval()` in `process_data/mimic-mortality.ipynb`. The edges are the 0,10,…,100th
percentiles, rounded and de-duplicated. A value `v` becomes `itemid|e[i-1]-e[i]`, or `itemid|e[-1]-` at or above
the maximum. Items with a single distinct value become just `itemid`.

In [ ]:
train_stays = set(cohort.loc[cohort['split'] == 'train', 'stay_id'])
samples = []
for p in tqdm(range(N_PARTITIONS), desc='sampling values'):
    df = read_partition(p, ['stay_id', 'itemid', 'value'])
    df = df[df['stay_id'].isin(train_stays)].sample(frac=1.0, random_state=SEED + p)
    samples.append(df[df.groupby('itemid').cumcount() < BUCKET_SAMPLES_PER_PARTITION][['itemid', 'value']])
samples = pd.concat(samples, ignore_index=True)

SCALE = 10_000
bucket_edges = {}
for item, vals in samples.groupby('itemid')['value']:
    vals = vals.to_numpy(dtype=np.float64)
    if np.unique(vals).size <= 1:
        bucket_edges[int(item)] = None
        continue
    q = np.percentile(vals, np.linspace(0, 100, N_BUCKETS + 1), method='lower')
    q = np.concatenate([[np.floor(q[0] * SCALE) / SCALE], np.round(q[1:-1], 4), [np.ceil(q[-1] * SCALE) / SCALE]])
    bucket_edges[int(item)] = np.unique(q)
del samples
print('items with buckets:', sum(e is not None for e in bucket_edges.values()), '| constant items:', sum(e is None for e in bucket_edges.values()))


def bucket_labels(item, edges):
    if edges is None or len(edges) == 1:
        return [f'{item}']
    return [f'{item}|{float(edges[i - 1])}-{float(edges[i])}' for i in range(1, len(edges))] + [f'{item}|{float(edges[-1])}-']


ITEM_LABELS = {item: np.array(bucket_labels(item, e)) for item, e in bucket_edges.items()}

## 4. Tokenizer

A word-level tokenizer over every possible bucket token for items seen in training, plus the demographic tokens.
Items that never appear in training map to `[UNK]`. The format is the same as `process_data/tokenizer.ipynb`.

In [ ]:
from tokenizers import Tokenizer, models

demo_tokens = sorted(set(cohort['gender_str']) | set(cohort['race_str']) | set(AGE_LABELS) | {'Gender/unknown', 'RACE/UNKNOWN'})
event_tokens = sorted({tok for labels in ITEM_LABELS.values() for tok in labels})
vocab_list = SPECIAL_TOKENS + demo_tokens + event_tokens
vocab = {tok: i for i, tok in enumerate(vocab_list)}
Tokenizer(models.WordLevel(vocab=vocab, unk_token='[UNK]')).save(str(OUT_DIR / 'tokenizer.json'))
print('vocab size:', len(vocab))

UNK_ID = vocab['[UNK]']
# per item: array of token ids aligned with ITEM_LABELS[item]
ITEM_TOKEN_IDS = {item: np.array([vocab[t] for t in labels], dtype=np.int32) for item, labels in ITEM_LABELS.items()}


def tokenize_events(df):
    """Adds a `token_id` column (vectorised per item)."""
    token_ids = np.full(len(df), UNK_ID, dtype=np.int32)
    items = df['itemid'].to_numpy()
    values = df['value'].to_numpy(dtype=np.float64)
    order = np.argsort(items, kind='stable')
    uniq, starts = np.unique(items[order], return_index=True)
    ends = np.append(starts[1:], len(order))
    for item, s, e in zip(uniq, starts, ends):
        idx = order[s:e]
        ids = ITEM_TOKEN_IDS.get(int(item))
        if ids is None:
            continue
        edges = bucket_edges[int(item)]
        if edges is None or len(edges) == 1:
            token_ids[idx] = ids[0]
        else:
            b = np.clip(np.searchsorted(edges, values[idx], side='right'), 1, len(edges))
            token_ids[idx] = ids[b - 1]
    return df.assign(token_id=token_ids)

## 5. Build sequences (+ LLM text for the test split)

In [ ]:
d_items = pd.read_csv(mimic_file('icu/d_items'), usecols=['itemid', 'abbreviation', 'label', 'unitname'])
d_lab = pd.read_csv(mimic_file('hosp/d_labitems'), usecols=['itemid', 'label', 'fluid'])
item_names = {int(r.itemid): (r.abbreviation if isinstance(r.abbreviation, str) else r.label) for r in d_items.itertuples()}
item_units = {int(r.itemid): r.unitname for r in d_items.itertuples() if isinstance(r.unitname, str)}
item_names.update({int(r.itemid): f'{r.label} ({r.fluid})' if isinstance(r.fluid, str) and r.fluid != 'Blood' else str(r.label)
                   for r in d_lab.itertuples()})

stay_info = cohort.set_index('stay_id')
cutoff_hours = ((stay_info['cutoff'] - stay_info['intime']).dt.total_seconds() / 3600).astype('float32')
llm_stays = set(cohort.loc[cohort['split'].isin(LLM_SPLITS), 'stay_id'])


def format_value(v):
    return f'{v:.4g}' if abs(v) < 1e6 else f'{v:.0f}'


def llm_text(df):
    """Serialise one partition's events like llm/tasks/mimic_mortality/data/gen_dataset.py:
    '<hours since ICU admission> h - <item>: <value> <unit>; ...' one line per timestamp."""
    df = df[df['stay_id'].isin(llm_stays)]
    df = df[df['hours'] >= df['stay_id'].map(cutoff_hours) - LLM_LOOKBACK_HOURS]
    if df.empty:
        return pd.DataFrame(columns=['stay_id', 'Text'])
    df = df.assign(bin=np.floor(df['hours'] * 60 / LLM_RESAMPLE_MINUTES))
    df = df.drop_duplicates(['stay_id', 'itemid', 'bin'], keep='first')  # sorted by time -> first value per bin
    df = df.groupby('stay_id').tail(LLM_MAX_ROWS)
    names = df['itemid'].map(item_names).fillna(df['itemid'].astype(str))
    units = df['itemid'].map(item_units).fillna('')
    df = df.assign(entry=(names + ': ' + df['value'].map(format_value) + ' ' + units).str.strip(),
                   t=df['hours'].round(2))
    lines = df.groupby(['stay_id', 't'], sort=True)['entry'].agg('; '.join).reset_index()
    lines['line'] = lines['t'].map(lambda h: f'{h:.2f} h') + ' - ' + lines['entry']
    return lines.groupby('stay_id')['line'].agg('\n'.join).rename('Text').reset_index()


SEQ_DIR = OUT_DIR / 'sequences'
shutil.rmtree(SEQ_DIR, ignore_errors=True)
SEQ_DIR.mkdir()
llm_parts = []
for p in tqdm(range(N_PARTITIONS), desc='building sequences'):
    df = read_partition(p)
    df = df.drop_duplicates().sort_values(['stay_id', 'hours', 'itemid'], kind='stable').reset_index(drop=True)
    df = tokenize_events(df)
    if len(llm_stays):
        llm_parts.append(llm_text(df))
    df = df.groupby('stay_id').tail(MAX_EVENTS)
    seq = df.groupby('stay_id').agg(input_ids=('token_id', list), times=('hours', list)).reset_index()
    seq.to_parquet(SEQ_DIR / f'part_{p:03d}.parquet', index=False)
    del df, seq
    gc.collect()

In [ ]:
seqs = pd.concat([pd.read_parquet(f) for f in sorted(SEQ_DIR.glob('*.parquet'))], ignore_index=True)
data = cohort.merge(seqs, on='stay_id', how='inner')
data['n_events'] = data['input_ids'].apply(len)
print('stays with no events in window:', len(cohort) - len(data), '| with <', MIN_EVENTS, 'events:', int((data['n_events'] < MIN_EVENTS).sum()))
data = data[data['n_events'] >= MIN_EVENTS].copy()
data['demo_ids'] = data.apply(lambda r: [vocab[r['gender_str']], vocab[r['race_str']], vocab[r['age_str']]], axis=1)
data['cutoff_hours'] = data['stay_id'].map(cutoff_hours)

for split in ['train', 'val', 'test']:
    part = data[data['split'] == split]
    part[['subject_id', 'stay_id', 'label', 'demo_ids', 'input_ids', 'times', 'n_events', 'cutoff_hours']].to_parquet(OUT_DIR / f'{split}.parquet', index=False)
    print(f'{split:>5}: {len(part):>6} stays | deaths {int(part["label"].sum()):>5} ({part["label"].mean():.2%}) | median events {int(part["n_events"].median())}')

In [ ]:
if llm_parts:
    llm = pd.concat(llm_parts, ignore_index=True).merge(data[['subject_id', 'stay_id', 'label', 'split', 'age_str', 'gender_str', 'race_str', 'cutoff_hours']], on='stay_id')
    header = ('Demographics: ' + llm['gender_str'].str.replace('Gender/', 'gender ') + ', '
              + llm['race_str'].str.replace('RACE/', 'race ').str.lower() + ', age ' + llm['age_str'].str.replace('Age/', ''))
    header += '\nPrediction time: ' + llm['cutoff_hours'].map(lambda h: f'{h:.2f}') + ' h after ICU admission'
    llm['Text'] = header + '\nTimeline (hours since ICU admission - measurements):\n' + llm['Text']
    llm['Label'] = llm['label'].astype(bool)
    for split in LLM_SPLITS:
        out = llm[llm['split'] == split][['subject_id', 'stay_id', 'Label', 'Text']].reset_index(drop=True)
        out.to_parquet(OUT_DIR / f'llm_{split}.parquet', index=False)
        print(f'llm_{split}.parquet: {len(out)} stays, median {int(out["Text"].str.len().median())} characters')
    print('\nExample:\n', llm['Text'].iloc[0][:1500])

Optionally remove the intermediate `events_raw/` (large) once everything above has been written:

In [ ]:
# shutil.rmtree(RAW_DIR)